# Chapter 10 — Run Provenance & Reproducibility (v2026)

> **LangChain 1.x / 2026 refresh.** Dual-mode secrets, optional LangSmith tracing, pinned core deps where applicable, and a standardized footer (Limitations & safety + cleanup + exercises). **REFACTORED_RUN_PROVENANCE_V2026**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare/blob/main/notebooks/CHDIR/FNAME)

## Learning objectives
- Capture corpus/index versions, prompts, and model/tool versions in a run manifest
- Record a retrieval/tool trace for auditability
- Export a self-contained reproducibility bundle

> **Runtime / cost / data.** Offline; no external calls. Writes a local run-manifest JSON.

## Environment setup

In [ ]:
import os

# Secrets are read from Colab Secrets if available, else from a local .env
try:
    from google.colab import userdata  # type: ignore

    def get_secret(name, default=""):
        return userdata.get(name) or default
except Exception:
    try:
        from dotenv import load_dotenv  # type: ignore

        load_dotenv()
    except Exception:
        pass

    def get_secret(name, default=""):
        return os.environ.get(name, default)

OPENAI_API_KEY = get_secret("LC4LS_OPENAI_API_KEY")
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# Optional LangSmith tracing (set LANGCHAIN_API_KEY to enable)
LANGSMITH_API_KEY = get_secret("LANGCHAIN_API_KEY", "")
LANGSMITH_PROJECT = "lc4lsh-chapter10-run-provenance"
if LANGSMITH_API_KEY.startswith(("lsv2_", "ls__")):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_PROJECT", LANGSMITH_PROJECT)
    print("LangSmith tracing ON ->", LANGSMITH_PROJECT)
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith tracing OFF")

## What is run provenance?

A **run manifest** is the single source of truth for *how* a result was produced. In an enterprise or regulated setting you must be able to answer: *which data, which model, which prompt, which tool versions produced this output?*

We build a manifest step by step:
1. **Environment** — package versions, python, timestamp.
2. **Corpus / index** — versions and hashes of the data used.
3. **Prompts & parameters** — exact prompt text and settings.
4. **Trace** — ordered retrieval/tool events.
5. **Export** — a portable JSON bundle.

In [ ]:
import hashlib, json, platform, sys, datetime

def sha(text):
    return hashlib.sha256(text.encode()).hexdigest()[:12]

# 1) Environment snapshot
import langchain
env_snapshot = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "langchain": getattr(langchain, "__version__", "unknown"),
    "captured_at": datetime.datetime.utcnow().isoformat() + "Z",
}
env_snapshot

In [ ]:
# 2) Corpus / index version (synthetic corpus)
CORPUS_DOCS = [
    {"id": "d1", "text": "Metformin is first-line for type 2 diabetes."},
    {"id": "d2", "text": "SGLT2 inhibitors reduce cardiovascular events."},
    {"id": "d3", "text": "GLP-1 agonists support weight reduction."},
]
corpus_version = sha(json.dumps([d["text"] for d in CORPUS_DOCS], sort_keys=True))
{"corpus_version": corpus_version, "n_docs": len(CORPUS_DOCS)}

In [ ]:
# 3) Prompts & parameters
PROMPT = "Answer using only the provided context. Question: {q}\nContext: {ctx}"
params = {"model": "gpt-4o-mini", "temperature": 0.0, "top_k": 2}
prompt_hash = sha(PROMPT)
{"prompt_hash": prompt_hash, **params}

In [ ]:
# 4) Simulated retrieval + generation trace
def retrieve(q, docs, k=2):
    toks = set(q.lower().split())
    scored = sorted(docs, key=lambda d: -len(toks & set(d["text"].lower().split())))
    return scored[:k]

trace = []
q = "Which drugs help cardiovascular outcomes?"
hits = retrieve(q, CORPUS_DOCS, k=params["top_k"])
for h in hits:
    trace.append({"event": "retrieval", "doc_id": h["id"], "snippet": h["text"][:40]})
answer = "SGLT2 inhibitors reduce cardiovascular events."
trace.append({"event": "generation", "model": params["model"], "answer": answer})
trace

In [ ]:
# 5) Assemble + export the run manifest
manifest = {
    "run_id": sha(q + prompt_hash + corpus_version),
    "question": q,
    "environment": env_snapshot,
    "corpus_version": corpus_version,
    "prompt_hash": prompt_hash,
    "parameters": params,
    "trace": trace,
    "answer": answer,
}
with open("run_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)
print(json.dumps(manifest, indent=2))

## Verifying reproducibility

To re-run deterministically, check the manifest's `corpus_version` and `prompt_hash` against the current values — a mismatch means drift.

In [ ]:
def verify(man):
    ok = man["corpus_version"] == sha(json.dumps([d["text"] for d in CORPUS_DOCS], sort_keys=True))
    ok &= man["prompt_hash"] == sha(PROMPT)
    return {"reproducible": bool(ok)}

print(verify(manifest))

## Limitations & safety

- **Enterprise / research-support only.** Human review is required before any production, clinical, or compliance decision.
- **Synthetic / de-identified data only.** Real PHI/PII requires governance and access controls.

In [ ]:
# Cleanup: drop references and free memory.
import gc

for _name in ["llm", "chain", "model", "agent", "app", "manifest", "report"]:
    globals().pop(_name, None)

gc.collect()
print("Cleanup complete.")

## Exercises

<details><summary>Q1. Why is a run manifest essential for reproducibility in an enterprise pipeline?</summary>
It captures corpus/index versions, prompts, and model/tool versions, so a past result can be audited, diffed, or re-run deterministically when models or data change.
</details>

<details><summary>Q2. Why treat guardrail / injection detections as drafts rather than automatic enforcement?</summary>
Detectors have false positives and negatives. In regulated settings, an error can block legitimate work or leak PHI, so a human confirms before enforcement.
</details>

<details><summary>Q3. Why compare frameworks by task and operations needs instead of popularity?</summary>
The best fit depends on state management, observability, deployment, and team skill — not download counts. A task-driven matrix makes the trade-offs explicit and defensible.
</details>

### Task A — Add a `git_sha` field to the manifest by calling `git rev-parse HEAD` (guard for non-git dirs).

### Task B — Extend the trace to log latency (ms) per event and summarize total retrieval vs generation time.

### Task C — Add a `dataset_row_hashes` list capturing each input row's hash, and verify one row.

### Task D — Write a `diff_manifest(a, b)` that reports which fields changed between two runs.